In [99]:
# pip install scikit-learn tqdm

In [100]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
#import tushare as ts
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from torch.utils.data import TensorDataset
from tqdm import tqdm
import torch.nn.functional as F
import scipy.io as scio
import time 

In [101]:
import scipy
data = scipy.io.loadmat('real2det(1).mat')
Dn=data['real2det']

In [102]:
def calculate_snr(clean_data, denoised_data):
    norm_clean = np.linalg.norm(clean_data)  
    norm_diff = np.linalg.norm(clean_data - denoised_data)
    
    snr = 20 * np.log10(norm_clean / norm_diff)
    return snr

In [103]:
import numpy as np

def patch2d(A, l1=12, l2=12, s1=s1, s2=s2, mode=1):
    [n1, n2] = A.shape
    if mode == 1:  # possible for other patching options
        tmp1 = np.mod(n1 - l1, s1)
        if tmp1 != 0:
            A = np.concatenate((A, np.zeros([s1 - tmp1, n2])), axis=0)
        tmp1 = 0
        tmp2 = np.mod(n2 - l2, s2)
        if tmp2 != 0:
            A = np.concatenate((A, np.zeros([A.shape[0], s2 - tmp2])), axis=1)
        tmp2 = 0
        [N1, N2] = A.shape
        X = []
        for i1 in range(0, N1 - l1 + 1, s1):
            for i2 in range(0, N2 - l2 + 1, s2):
                tmp = np.reshape(A[i1:i1+l1, i2:i2+l2], (l1*l2, 1), order='F')
                X.append(tmp)
        X = np.array(X)
    else:
        pass
    return X[:, :, 0]

def patch2d_inv(X, n1, n2, l1=12, l2=12, s1=s1, s2=s2, mode=1):
    if mode == 1:  # possible for other patching options
        tmp1 = np.mod(n1 - l1, s1)
        tmp2 = np.mod(n2 - l2, s2)
        if tmp1 != 0 and tmp2 != 0:
            A = np.zeros([n1 + s1 - tmp1, n2 + s2 - tmp2])
            mask = np.zeros([n1 + s1 - tmp1, n2 + s2 - tmp2])
        if tmp1 == 0 and tmp2 != 0:
            A = np.zeros([n1, n2 + s2 - tmp2])
            mask = np.zeros([n1, n2 + s2 - tmp2])
        if tmp1 != 0 and tmp2 == 0:
            A = np.zeros([n1 + s1 - tmp1, n2])
            mask = np.zeros([n1 + s1 - tmp1, n2])
        if tmp1 == 0 and tmp2 == 0:
            A = np.zeros([n1, n2])
            mask = np.zeros([n1, n2])
        
        [N1, N2] = A.shape
        id = -1
        for i1 in range(0, N1 - l1 + 1, s1):
            for i2 in range(0, N2 - l2 + 1, s2):
                id = id + 1
                A[i1:i1+l1, i2:i2+l2] = A[i1:i1+l1, i2:i2+l2] + np.reshape(X[id, :], [l1, l2], order='F')
                mask[i1:i1+l1, i2:i2+l2] = mask[i1:i1+l1, i2:i2+l2] + np.ones([l1, l2])
        
        A = A / mask
        A = A[0:n1, 0:n2]
    else:
        pass
    return A

In [104]:
w1=12
w2=12
s1=1
s2=1
X=patch2d(Dn,w1,w2,s1,s2)
dz=1
dx=1
[nz,nx]=Dn.shape
X.shape
print(X.dtype)
X = X.astype(np.float32)
Pdata = torch.from_numpy(X)
print(Pdata.dtype)

float64
torch.float32


In [105]:
# Data split: 80% training data, 20% validation data
train_size = int(len(Pdata) * 0.8)
train = Pdata[:train_size]
valid = Pdata[train_size:]

# Print the shapes of the training set and validation set
print(f"Train shape: {train.shape}")
print(f"Valid shape: {valid.shape}")
# Create TensorDataset
train_data = TensorDataset(train)
valid_data = TensorDataset(valid)

batch_size1 = 64
# Create DataLoader
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size1, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_data, batch_size=batch_size1, shuffle=True)

Train shape: torch.Size([46893, 144])
Valid shape: torch.Size([11724, 144])


In [106]:
train.shape
valid.shape
print(valid.dtype)

torch.float32


In [107]:
#Fully connected (FC) block
class FCB(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.3):
        super().__init__()
        
        self.fc = nn.Linear(input_size, output_size)
        self.activation = nn.LeakyReLU()
        self.bn = nn.BatchNorm1d(output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc(x)
        x = self.activation(x) 
        x = self.bn(x)
        x = self.dropout(x)
        
        return x

In [108]:
class Encoder(nn.Module):
    def __init__(self, input_size, dropout=0.3):
        super().__init__()


        self.fcb0 = FCB(input_size, 128, dropout)
        self.fcb1 = FCB(128, 64, dropout)

        self.fcb2 = FCB(64, 32, dropout)

        self.fcb3 = FCB(32, 16, dropout)

        self.fcb4 = FCB(16, 8, dropout)

        self.fcb5 = FCB(8, 4, dropout)

        self.fcb6 = FCB(4, 4, dropout)

        


    def forward(self, x):

        #x = x.reshape(x.shape[0],1,x.shape[1])

        x1 = self.fcb0(x)#x->128
        x2 = self.fcb1(x1)

        x3 = self.fcb2(x2)

        x4 = self.fcb3(x3)

        x5 = self.fcb4(x4)

        x6 = self.fcb5(x5)

        x7 = self.fcb6(x6)

        x8 = self.fcb6(x7)
        

        

        return x2,x3,x4,x5,x6,x7,x8

In [109]:
class Decoder(nn.Module):
    def __init__(self, output_size, dropout=0.3):
        super().__init__()
        self.pab1 = FCB(8, 8, dropout)
        self.pab2 = FCB(16, 16, dropout)
        self.pab3 = FCB(32, 32, dropout)
        self.pab4 = FCB(64, 64, dropout)
        self.pab5 = FCB(128, 128, dropout)
        self.pab6 = FCB(128, output_size, dropout)

        self.activation = nn.Identity()


    def forward(self,x2,x3,x4,x5,x6,x7,x8):
        
        x = torch.cat((x6,x8),dim=1)
        x = self.pab1(x)
        x = torch.cat((x,x5),dim=1)
        x = self.pab2(x)
        x = torch.cat((x,x4),dim=1)
        x = self.pab3(x)
        x = torch.cat((x,x3),dim=1)
        x = self.pab4(x)
        x = torch.cat((x,x2),dim=1)
        x = self.pab5(x)
        x = self.pab6(x)

        x = self.activation(x)
        return x

In [110]:
class AutoEncoder(nn.Module):
    def __init__(self,input_size, output_size):
        super().__init__()
        self.encoder = Encoder(input_size)
        self.decoder = Decoder(output_size)

    def forward(self, x):
        x2,x3,x4,x5,x6,x7,x8= self.encoder(x)
        x = self.decoder(x2,x3,x4,x5,x6,x7,x8)

        return x

In [112]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoEncoder(w1*w2, w1*w2).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

In [113]:
start_time = time.time()
es_cnt = 0
es_thres = 5
prev_train_loss = float('inf')
loss_train = []
loss_valid = []
num_epochs = 100 
#num_batch_train = 0
for epoch in range(num_epochs):
  #train_bar = tqdm(train_loader)
  train_loss = 0.0
  
  for i , (batch) in enumerate(train_loader):

    # turn to device
    train_batch = batch[0].to(device)
    
    # train  
    optimizer.zero_grad()
    outputs = model(train_batch)
    loss = criterion(outputs, train_batch)
    loss.backward() 
    optimizer.step()
    
    train_loss += loss.item()
    
  train_loss = train_loss/(np.ceil(train.size(0)/batch_size1))
  loss_train.append(train_loss)
    



  
  valid_loss = 0.0
  #num_batch_vaild = 0
  with torch.no_grad():
    for i , (batch) in enumerate(valid_loader):
    #for batch in vaild_loader:
    
      val_batch = batch[0].to(device)
      
      outputs = model(val_batch)
      loss = criterion(outputs, val_batch)
      valid_loss += loss.item()
    valid_loss = valid_loss/(np.ceil(valid.size(0)/batch_size1))
    loss_valid.append(valid_loss)
    print("Epoch [{}/{}], Train Loss: {:.4f}, Valid Loss: {:.4f}".format(epoch+1, num_epochs, train_loss, valid_loss))

    
    # Early stopping
    if train_loss - prev_train_loss >= 0:
        es_cnt += 1
    else:
        #es_cnt = 0
        pass

    if es_cnt >= es_thres:
        print(f"Early stopped at epoch {epoch}, train loss stop improving")
        break  

    prev_train_loss = train_loss



   
print("Training finished")
current_time = time.time()
time_sum = current_time-start_time
print(time_sum)

Epoch [1/100], Train Loss: 1.6169, Valid Loss: 0.8752
Epoch [2/100], Train Loss: 1.2836, Valid Loss: 0.8733
Epoch [3/100], Train Loss: 1.2387, Valid Loss: 0.8669
Epoch [4/100], Train Loss: 1.2213, Valid Loss: 0.8696
Epoch [5/100], Train Loss: 1.2074, Valid Loss: 0.8610
Epoch [6/100], Train Loss: 1.1896, Valid Loss: 0.8580
Epoch [7/100], Train Loss: 1.1758, Valid Loss: 0.8500
Epoch [8/100], Train Loss: 1.1662, Valid Loss: 0.8577
Epoch [9/100], Train Loss: 1.1610, Valid Loss: 0.8509
Epoch [10/100], Train Loss: 1.1582, Valid Loss: 0.8507
Epoch [11/100], Train Loss: 1.1490, Valid Loss: 0.8416
Epoch [12/100], Train Loss: 1.1472, Valid Loss: 0.8409
Epoch [13/100], Train Loss: 1.1405, Valid Loss: 0.8483
Epoch [14/100], Train Loss: 1.1424, Valid Loss: 0.8423
Epoch [15/100], Train Loss: 1.1391, Valid Loss: 0.8428
Epoch [16/100], Train Loss: 1.1383, Valid Loss: 0.8391
Epoch [17/100], Train Loss: 1.1334, Valid Loss: 0.8432
Epoch [18/100], Train Loss: 1.1317, Valid Loss: 0.8368
Epoch [19/100], Tra

In [114]:
loss_train = pd.DataFrame(loss_train)
loss_valid = pd.DataFrame(loss_valid)

loss = pd.concat([loss_train,loss_valid],axis=1)

In [115]:
loss.columns = ['train_loss','vaild_loss']

In [116]:
torch.save(model.state_dict(), r'.\model_2dreal_patch.pth')

In [117]:
model = AutoEncoder(w1*w2,w1*w2).to(device)
Pdata = Pdata.to(device)
model.load_state_dict(torch.load(r'.\model_2dreal_patch.pth'))
model.eval()
with torch.no_grad():
    output = model(Pdata)
    loss = criterion(output, Pdata)
output = output.cpu()
output = output.numpy()
# ========== Reconstruct denoised data ==========
# Reconstruct the denoised data
n1 = Dn.shape[0]  # MATLAB：n1 = size(Dn, 1)
n2 = Dn.shape[1]  # MATLAB：n2 = size(Dn, 2)
D_denoised = patch2d_inv(output,n1,n2,w1,w2,s1,s2)
scipy.io.savemat(r"PatchUnet real2d result6.mat", 
        {'D_denoised': D_denoised})  